# AttentionForge Studio: Raw PyTorch Transformer Core

This notebook shows the model core behind the full-stack AttentionForge Studio app.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import torch
import matplotlib.pyplot as plt

from backend.attentionforge.model import MiniTransformerLM
from backend.attentionforge.tokenizer import DEFAULT_CORPUS, WordTokenizer

torch.manual_seed(42)

tokenizer = WordTokenizer.from_text(DEFAULT_CORPUS)
model = MiniTransformerLM(
    vocab_size=tokenizer.vocab_size,
    d_model=192,
    num_heads=6,
    num_layers=6,
    ff_hidden_dim=768,
    max_len=96,
    dropout=0.0,
)
model.eval()

print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")


In [ ]:
ids = tokenizer.safe_encode("attention lets each token decide which earlier tokens matter")
idx = torch.tensor([ids], dtype=torch.long)

logits, diagnostics = model(idx, causal_mask=True)

print("Input shape:", idx.shape)
print("Logits shape:", logits.shape)
print("Attention shape:", diagnostics[0]["attention_weights"].shape)


In [ ]:
attention = diagnostics[0]["attention_weights"][0, 0].detach()

plt.figure(figsize=(6, 5))
plt.imshow(attention)
plt.title("Layer 0 / Head 0 attention")
plt.xlabel("Key token position")
plt.ylabel("Query token position")
plt.colorbar()
plt.show()


Run the full app:

```bash
python -m uvicorn backend.main:app --reload --port 8000
cd frontend
npm install
npm run dev
```